In [1]:
from pyspark.sql import SparkSession as Spksess,functions as F ,types as T,DataFrame as DFT
from pyspark import SparkConf,StorageLevel, SparkContext
from pyspark.errors import PySparkException as PyEx
from pyspark.sql.utils import AnalysisException as AnyEx
import sys,os,re
from datetime import datetime as dt
from pyspark.sql.functions import col, sum as spark_sum, col, count, when, isnan, isnull
from pyspark.sql.types import *
import pandas as pd 
import traceback

In [2]:
sparkconfiguration  = SparkConf()

In [3]:
sparkconfiguration.set("spark.app.name","John_MicrosoftSparkJob-ML") 
sparkconfiguration.set("spark.master", "local[*]")                
# sparkconfiguration.set("spark.driver.memory", "1g")      
# sparkconfiguration.set("spark.driver.cores", "1")                 
# sparkconfiguration.set("spark.ui.port", "4040")                
# sparkconfiguration.set("spark.executor.memory", "3g")      
# sparkconfiguration.set("spark.executor.cores", "2")     
# sparkconfiguration.set("spark.executor.memoryOverhead", "1g")          
# sparkconfiguration.set("spark.executor.instances", "1")   
sparkconfiguration.set("spark.shuffle.io.retryWait", "180s")      
sparkconfiguration.set("spark.default.parallelism","27")         
# sparkconfiguration.set("spark.sql.shuffle.partitions","30")       
# sparkconfiguration.set("spark.task.cpus", "1")  
# sparkconfiguration.set("spark.memory.fraction","0.8")            
# sparkconfiguration.set("spark.memory.storageFraction","0.5")  
sparkconfiguration.set("spark.driver.memory", "2g")     
sparkconfiguration.set("spark.executor.memory", "4g")   
sparkconfiguration.set("spark.executor.memoryOverhead", "512m")  
sparkconfiguration.set("spark.sql.files.maxPartitionBytes", "67108864")  
sparkconfiguration.set("spark.sql.shuffle.partitions", "50")  
sparkconfiguration.set("spark.memory.fraction", "0.8")   
sparkconfiguration.set("spark.memory.storageFraction", "0.3")    
sparkconfiguration.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")  
sparkconfiguration.set("spark.kryo.registrationRequired","false")  
sparkconfiguration.set("spark.kryo.classesToRegister", "org.apache.spark.sql.Row")  
sparkconfiguration.set("spark.eventLog.enabled", "true")           
sparkconfiguration.set("spark.eventLog.dir", "/home/john/spark_event_logs/")  
sparkconfiguration.set("spark.history.fs.logDirectory","/home/john/spark_history_logs/")  
sparkconfiguration.set("spark.jars", "/home/john/spark_jar_files/postgresql-42.7.7.jar")
sparkconfiguration.set("spark.sql.adaptive.enabled", "true")
sparkconfiguration.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
sparkconfiguration.set("spark.local.dir", "/home/john/volume_folder/")  


In [4]:
SparkSession_init = Spksess.builder.config(conf=sparkconfiguration).getOrCreate()

25/07/27 17:59:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/27 17:59:27 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [5]:
database_config_url      = "jdbc:postgresql://{}:{}/{}".format('host.docker.internal', '5432', 'dba_microsoft_cbs_da')
db_credential_properties = {"user": 'john_user',"password": 'abc@12345',"driver": 'org.postgresql.Driver'}

In [6]:
def capture_spark_error(func):
    """
    Decorator function to capture detailed error information from Spark Operations and general exceptions.
    """
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except (AnyEx, PyEx) as e:
            error_info = {
                'success': False,
                'error_type': type(e).__name__,
                'error_message': str(e),
                'error_args': e.args,
                'timestamp': dt.now().isoformat()
            }
            exc_type, exc_value, exc_traceback = sys.exc_info()
            error_info.update({
                'traceback': traceback.format_exc(),
                'exception_type': exc_type,
                'exception_value': exc_value,
                'line_number': exc_traceback.tb_lineno if exc_traceback else None
            })
            return error_info
        except Exception as e:
            # Catch any other exception
            error_info = {
                'success': False,
                'error_type': type(e).__name__,
                'error_message': str(e),
                'error_args': e.args,
                'timestamp': dt.now().isoformat()
            }
            exc_type, exc_value, exc_traceback = sys.exc_info()
            error_info.update({
                'traceback': traceback.format_exc(),
                'exception_type': exc_type,
                'exception_value': exc_value,
                'line_number': exc_traceback.tb_lineno if exc_traceback else None
            })
            return error_info
    return wrapper

In [7]:
absolute_folder_path = "/home/john/program_data"
files = [os.path.join(absolute_folder_path, f)for f in os.listdir(absolute_folder_path) if os.path.isfile(os.path.join(absolute_folder_path, f))]

In [8]:
@capture_spark_error
def read_csv_files(file_path: str) -> DFT:
    df = SparkSession_init.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    return df


In [9]:
test_input_Data = read_csv_files(files[0])
train_input_Data = read_csv_files(files[1])


In [10]:
print(test_input_Data.dtypes)
print(train_input_Data.dtypes)

[('Id', 'bigint'), ('OrgId', 'int'), ('IncidentId', 'int'), ('AlertId', 'int'), ('Timestamp', 'timestamp'), ('DetectorId', 'int'), ('AlertTitle', 'int'), ('Category', 'string'), ('MitreTechniques', 'string'), ('IncidentGrade', 'string'), ('ActionGrouped', 'string'), ('ActionGranular', 'string'), ('EntityType', 'string'), ('EvidenceRole', 'string'), ('DeviceId', 'int'), ('Sha256', 'int'), ('IpAddress', 'int'), ('Url', 'int'), ('AccountSid', 'int'), ('AccountUpn', 'int'), ('AccountObjectId', 'int'), ('AccountName', 'int'), ('DeviceName', 'int'), ('NetworkMessageId', 'int'), ('EmailClusterId', 'double'), ('RegistryKey', 'int'), ('RegistryValueName', 'int'), ('RegistryValueData', 'int'), ('ApplicationId', 'int'), ('ApplicationName', 'int'), ('OAuthApplicationId', 'int'), ('ThreatFamily', 'string'), ('FileName', 'int'), ('FolderPath', 'int'), ('ResourceIdName', 'int'), ('ResourceType', 'string'), ('Roles', 'string'), ('OSFamily', 'int'), ('OSVersion', 'int'), ('AntispamDirection', 'string')

In [11]:
test_missing_columns = set(test_input_Data.columns) - set(train_input_Data.columns)
train_missing_columns = set(train_input_Data.columns) - set(test_input_Data.columns)
print("Missing columns in test:",test_missing_columns )
print("Missing columns in train:",train_missing_columns)


Missing columns in test: {'Usage'}
Missing columns in train: set()


In [12]:

test_input_Data[['Usage']].groupBy('Usage').count().show()


+-------+-------+
|  Usage|  count|
+-------+-------+
|Private|1232555|
| Public|2915437|
+-------+-------+



In [13]:
@capture_spark_error
def  append_missing_columns_dft (input_df: DFT, missing_columns: set, comparison_dft: DFT) -> DFT:
    for column in missing_columns:
        comparision_dtypes = comparison_dft.schema[column].dataType if column in comparison_dft.columns else T.StringType()
        input_df = input_df.withColumn(column, F.lit(None).cast(comparision_dtypes))
    return input_df

In [14]:
test_missing_columns_udft = append_missing_columns_dft(test_input_Data,train_missing_columns, train_input_Data)
train_missing_columns_udft = append_missing_columns_dft(train_input_Data, test_missing_columns, test_input_Data)
print("Missing columns in test:",set(test_missing_columns_udft.columns) - set(train_missing_columns_udft.columns) )
print("Missing columns in train:",set(train_missing_columns_udft.columns) - set(test_missing_columns_udft.columns))

Missing columns in test: set()
Missing columns in train: set()


In [15]:
@capture_spark_error
def count_of_nulls_column(df):
    # Get null counts as a dictionary
    null_counts_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    data = [(col_name, null_count,str(null_count/1000)+'k') for col_name, null_count in null_counts_row.items() if null_count > 0 ]
    # Create a new DataFrame with two columns: column_name and null_count
    return SparkSession_init.createDataFrame(data, ["column_name", "null_count", "null_countink"])

# Example usage:


In [16]:
test_nulls_columsn_list  = count_of_nulls_column(test_missing_columns_udft)
test_nulls_columsn_list.show(test_nulls_columsn_list.count(),truncate=False)

25/07/15 21:14:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+----------+-------------+
|column_name      |null_count|null_countink|
+-----------------+----------+-------------+
|MitreTechniques  |2307104   |2307.104k    |
|ActionGrouped    |4146079   |4146.079k    |
|ActionGranular   |4146079   |4146.079k    |
|EmailClusterId   |4106285   |4106.285k    |
|ThreatFamily     |4116614   |4116.614k    |
|ResourceType     |4144998   |4144.998k    |
|Roles            |4039317   |4039.317k    |
|AntispamDirection|4071481   |4071.481k    |
|SuspicionLevel   |3498157   |3498.157k    |
|LastVerdict      |3155260   |3155.26k     |
+-----------------+----------+-------------+



In [17]:
train_nulls_columsn_list  = count_of_nulls_column(train_missing_columns_udft)
train_nulls_columsn_list.show(train_nulls_columsn_list.count(),truncate=False)

+-----------------+----------+-------------+
|column_name      |null_count|null_countink|
+-----------------+----------+-------------+
|MitreTechniques  |5468386   |5468.386k    |
|IncidentGrade    |51340     |51.34k       |
|ActionGrouped    |9460773   |9460.773k    |
|ActionGranular   |9460773   |9460.773k    |
|EmailClusterId   |9420025   |9420.025k    |
|ThreatFamily     |9441956   |9441.956k    |
|ResourceType     |9509762   |9509.762k    |
|Roles            |9298686   |9298.686k    |
|AntispamDirection|9339535   |9339.535k    |
|SuspicionLevel   |8072708   |8072.708k    |
|LastVerdict      |7282572   |7282.572k    |
|Usage            |9516837   |9516.837k    |
+-----------------+----------+-------------+



In [18]:
# Perform a full outer join on 'column_name'
nulls_comparison = train_nulls_columsn_list.alias("train").join(
    test_nulls_columsn_list.alias("test"),
    on="column_name",
    how="full_outer"
).select(
    "column_name",
    F.coalesce(F.col("train.null_count"),F.lit(0)).alias("train_null_count"),
    F.coalesce(F.col("test.null_count"),F.lit(0)).alias("test_null_count")
)


# Add a column to indicate which DataFrame has more nulls
nulls_comparison = nulls_comparison.withColumn(
    "more_nulls_in",
    F.when(
        F.col("train_null_count") > F.col("test_null_count"), F.lit("train")
    ).when(
        F.col("test_null_count") > F.col("train_null_count"), F.lit("test")
    ).otherwise(F.lit("equal"))
)

nulls_comparison.show(nulls_comparison.count(), truncate=False)

+-----------------+----------------+---------------+-------------+
|column_name      |train_null_count|test_null_count|more_nulls_in|
+-----------------+----------------+---------------+-------------+
|ActionGranular   |9460773         |4146079        |train        |
|ActionGrouped    |9460773         |4146079        |train        |
|AntispamDirection|9339535         |4071481        |train        |
|EmailClusterId   |9420025         |4106285        |train        |
|IncidentGrade    |51340           |0              |train        |
|LastVerdict      |7282572         |3155260        |train        |
|MitreTechniques  |5468386         |2307104        |train        |
|ResourceType     |9509762         |4144998        |train        |
|Roles            |9298686         |4039317        |train        |
|SuspicionLevel   |8072708         |3498157        |train        |
|ThreatFamily     |9441956         |4116614        |train        |
|Usage            |9516837         |0              |train     

In [48]:
combined_df[['Usage']].groupBy('Usage').count().show()


25/07/09 22:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/09 22:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/09 22:17:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/09 22:17:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------+
|  Usage|  count|
+-------+-------+
|Private|5990973|
| Public|7673856|
+-------+-------+



In [19]:
updated_combined_dft = train_missing_columns_udft.unionByName(test_missing_columns_udft)
repartition_count = int(3458 / 128)  
updated_combined_dft.repartition(repartition_count)
updated_combined_dft[['Usage']].groupBy('Usage').count().show()

+-------+-------+
|  Usage|  count|
+-------+-------+
|   NULL|9516837|
|Private|1232555|
| Public|2915437|
+-------+-------+



In [20]:
# Step 1: Read the table into a DataFrame
duplicate_table_df = SparkSession_init.read.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_duplicates",  # replace with your table name
    properties=db_credential_properties
)

# Step 2: Get all columns from the new DataFrame
join_columns = duplicate_table_df.columns



In [21]:
from pyspark.sql.functions import coalesce, lit, col,date_format

# Set aliases
combined_df_alias = updated_combined_dft.alias("a")
duplicate_table_df_alias = duplicate_table_df.alias("b")

def get_coalesce_expr( df, colname,alias= None):
    column = f"{alias}.{colname}"if alias is not None else colname
    dtype = dict(df.dtypes)[colname]
    if dtype in ['int', 'bigint', 'double', 'float', 'decimal']:
        return coalesce(col(column), lit(0))
    elif dtype in ['string']:
        return coalesce(col(column), lit(''))
    elif dtype in ['date', 'timestamp']:
        return coalesce(
            date_format(col(column), "ddMMyyyyHHmmss").cast("long"),
            lit(1010001000000).cast("long")
        )
    else:
        return col(column)

# Build join condition using aliases
join_cond = [
    get_coalesce_expr( updated_combined_dft, c,"a") == get_coalesce_expr(duplicate_table_df, c,"b")
    for c in join_columns if not(c == 'count_value')
]

select_columns = [ col(f"{'a'}.{c}").alias(c)   for c in combined_df_alias.columns]

# Perform anti-join
result_df_without_duplicates = combined_df_alias.join(duplicate_table_df_alias, on=join_cond, how='left_anti')
result_df_with_duplicates = combined_df_alias.join(duplicate_table_df_alias, on=join_cond, how='inner')

# Now select only left table columns (from alias 'a')
result_df_with_duplicates = result_df_with_duplicates.select(
    [col(f"a.{c}").alias(c) for c in combined_df_alias.columns]
)

result_df_without_duplicates = combined_df_alias.join(duplicate_table_df_alias, on=join_cond, how='left_anti')
# Now select only left table columns (from alias 'a')
result_df_without_duplicates = result_df_without_duplicates.select(
    [col(f"a.{c}").alias(c) for c in combined_df_alias.columns]
)

In [22]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

order_by_columns  = [get_coalesce_expr(result_df_with_duplicates, c) for c in result_df_with_duplicates.columns]

# Add a row_number column
dup_dft = result_df_with_duplicates.withColumn("dup_rank", row_number().over(Window.partitionBy(["Id","OrgId","IncidentId","AlertId","AccountSid","AccountObjectId","AccountName","NetworkMessageId","ApplicationId"]).orderBy(order_by_columns)))
dup_dft.repartition(repartition_count)
# # Filter to keep only records with rank == 1
deduped_dft = dup_dft.filter(col("dup_rank") == 1).drop("rank")
deduped_dft = deduped_dft.drop("dup_rank")
deduped_dft.repartition(repartition_count)
duplicate_fixed_dft = result_df_without_duplicates.unionByName(deduped_dft, allowMissingColumns=True)
duplicate_fixed_dft

DataFrame[Id: bigint, OrgId: int, IncidentId: int, AlertId: int, Timestamp: timestamp, DetectorId: int, AlertTitle: int, Category: string, MitreTechniques: string, IncidentGrade: string, ActionGrouped: string, ActionGranular: string, EntityType: string, EvidenceRole: string, DeviceId: int, Sha256: int, IpAddress: int, Url: int, AccountSid: int, AccountUpn: int, AccountObjectId: int, AccountName: int, DeviceName: int, NetworkMessageId: int, EmailClusterId: double, RegistryKey: int, RegistryValueName: int, RegistryValueData: int, ApplicationId: int, ApplicationName: int, OAuthApplicationId: int, ThreatFamily: string, FileName: int, FolderPath: int, ResourceIdName: int, ResourceType: string, Roles: string, OSFamily: int, OSVersion: int, AntispamDirection: string, SuspicionLevel: string, LastVerdict: string, CountryCode: int, State: int, City: int, Usage: string]

In [25]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, when

w = Window().orderBy(F.monotonically_increasing_id())
duplicate_fixed_dft = duplicate_fixed_dft.withColumn("row_num", row_number().over(w))
duplicate_fixed_dft_up = duplicate_fixed_dft.withColumn(
    "Usage",
    when((F.col("row_num") % 2 == 1), "Public").otherwise("Private")
).drop("row_num")

In [27]:
duplicate_fixed_dft_up.write.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_dups",  # replace with your table name
    properties=db_credential_properties,
    mode="overwrite"  # or "append" based on your requirement
)

25/07/15 22:11:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 2

In [35]:
duplicate_fixed_dft_up.drop("MitreTechniques")

rename_dict = {
"\"Id\"":"id",
"\"OrgId\"":"org_id",
"\"IncidentId\"":"incident_id",
"\"AlertId\"":"alert_id",
"\"Timestamp\"":"creation_timestamp",
"\"DetectorId\"":"detector_id",
"\"AlertTitle\"":"alert_title",
"\"Category\"":"category",
"\"IncidentGrade\"":"incident_grade",
"\"ActionGrouped\"":"action_grouped",
"\"ActionGranular\"":"action_granular",
"\"EntityType\"":"entity_type",
"\"EvidenceRole\"":"evidence_role",
"\"DeviceId\"":"device_id",
"\"Sha256\"":"sha256",
"\"IpAddress\"":"ip_address",
"\"Url\"":"url",
"\"AccountSid\"":"account_sid",
"\"AccountUpn\"":"account_upn",
"\"AccountObjectId\"":"account_object_id",
"\"AccountName\"":"account_name",
"\"DeviceName\"":"device_name",
"\"NetworkMessageId\"":"network_message_id",
"\"EmailClusterId\"":"email_cluster_id",
"\"RegistryKey\"":"registry_key",
"\"RegistryValueName\"":"registry_value_name",
"\"RegistryValueData\"":"registry_value_data",
"\"ApplicationId\"":"application_id",
"\"ApplicationName\"":"application_name",
"\"OAuthApplicationId\"":"oauth_application_id",
"\"ThreatFamily\"":"threat_family",
"\"FileName\"":"file_name",
"\"FolderPath\"":"folder_path",
"\"ResourceIdName\"":"resource_id_name",
"\"ResourceType\"":"resource_type",
"\"Roles\"":"roles",
"\"OSFamily\"":"os_family",
"\"OSVersion\"":"os_version",
"\"AntispamDirection\"":"anti_spam_direction",
"\"SuspicionLevel\"":"suspicion_level",
"\"LastVerdict\"":"last_verdict",
"\"CountryCode\"":"country_code",
"\"State\"":"state",
"\"City\"":"city",
"\"Usage\"":"usage",
}
final_microsft_data = duplicate_fixed_dft_up

final_microsft_data = final_microsft_data.select(
    *[col(c).alias(rename_dict.get(f'"{c}"', c)) for c in final_microsft_data.columns]
)
repartition_count = int(final_microsft_data.count() / 128) if final_microsft_data.count() > 128 else 1
final_microsft_data.repartition(repartition_count)


DataFrame[id: bigint, org_id: int, incident_id: int, alert_id: int, creation_timestamp: timestamp, detector_id: int, alert_title: int, category: string, MitreTechniques: string, incident_grade: string, action_grouped: string, action_granular: string, entity_type: string, evidence_role: string, device_id: int, sha256: int, ip_address: int, url: int, account_sid: int, account_upn: int, account_object_id: int, account_name: int, device_name: int, network_message_id: int, email_cluster_id: double, registry_key: int, registry_value_name: int, registry_value_data: int, application_id: int, application_name: int, oauth_application_id: int, threat_family: string, file_name: int, folder_path: int, resource_id_name: int, resource_type: string, roles: string, os_family: int, os_version: int, anti_spam_direction: string, suspicion_level: string, last_verdict: string, country_code: int, state: int, city: int, usage: string]

In [36]:
final_microsft_data.write.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_data",  # replace with your table name
    properties=db_credential_properties,
    mode="overwrite"  )

25/07/15 23:05:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:05:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:05:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:06:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:06:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:06:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 2

In [10]:
# ML_data_input = SparkSession_init.read.jdbc(
#     url=database_config_url,
#     table="staging_input_data.microsoft_cyber_data",  # replace with your table name
#     properties=db_credential_properties
# )

ML_data_input = SparkSession_init.read \
    .format("jdbc") \
    .option("url", database_config_url) \
    .option("dbtable", "staging_input_data.microsoft_cyber_data") \
    .option("numPartitions", "30") \
    .option("fetchsize", "150000") \
    .option("maxPartitionBytes", "67108864") \
    .options(**db_credential_properties) \
    .load()

# Drop column and repartition efficiently
ML_data_input = ML_data_input.drop("mitre_techniques")
repartition_count = int(6000 / 128)
ML_data_input = ML_data_input.repartition(repartition_count)

# Persist only if you need to reuse ML_data_input multiple times
ML_data_input.persist(StorageLevel.DISK_ONLY)

# Show a sample to verify
ML_data_input.show(5)

+-------------+------+-----------+--------+-------------------+-----------+-----------+-----------------+--------------+--------------+---------------+-----------+-------------+---------+------+----------+------+-----------+-----------+-----------------+------------+-----------+------------------+----------------+------------+-------------------+-------------------+--------------+----------------+--------------------+-------------+---------+-----------+----------------+-------------+-----------+---------+----------+-------------------+---------------+------------+------------+-----+-----+-------+
|           id|org_id|incident_id|alert_id| creation_timestamp|detector_id|alert_title|        catergory|incident_grade|action_grouped|action_granular|entity_type|evidence_role|device_id|sha256|ip_address|   url|account_sid|account_upn|account_object_id|account_name|device_name|network_message_id|email_cluster_id|registry_key|registry_value_name|registry_value_data|application_id|application_n

In [8]:
output_path = "/home/john/volume_folder/Ml_data_test.parquet"
ML_data_input.write.parquet(output_path)

25/07/26 18:59:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/26 18:59:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/26 18:59:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/26 18:59:39 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/26 18:59:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/26 18:59:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/26 18:59:47 WARN MemoryManager: Total allocation exceeds 95.00% 

In [9]:
ML_data_input.count()

13630983

In [14]:
output_path = "/home/john/volume_folder/Ml_data_test.parquet"
ML_data_input.write.parquet(output_path)

NameError: name 'ML_data_input' is not defined

In [7]:
# output_path = "/home/john/volume_folder/Ml_data_test.parquet"
# ML_data_input.write.parquet(output_path)

ML_data_parquet = SparkSession_init.read \
                    .format("parquet").load("/home/john/volume_folder/Ml_data_test.parquet")


In [8]:
ML_data_parquet = ML_data_parquet.coalesce(20)
ML_data_parquet.show(5)

25/07/27 17:59:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+------+-----------+--------+-------------------+-----------+-----------+------------------+--------------+--------------+---------------+-----------+-------------+---------+------+----------+------+-----------+-----------+-----------------+------------+-----------+------------------+----------------+------------+-------------------+-------------------+--------------+----------------+--------------------+-------------+---------+-----------+----------------+-------------+-----+---------+----------+-------------------+---------------+------------+------------+-----+-----+-------+
|           id|org_id|incident_id|alert_id| creation_timestamp|detector_id|alert_title|         catergory|incident_grade|action_grouped|action_granular|entity_type|evidence_role|device_id|sha256|ip_address|   url|account_sid|account_upn|account_object_id|account_name|device_name|network_message_id|email_cluster_id|registry_key|registry_value_name|registry_value_data|application_id|application_name|

In [9]:
from pyspark.sql.types import FloatType, DoubleType, StringType, TimestampType, DateType

def is_missing(col_name, dtype):
    if isinstance(dtype, (FloatType, DoubleType)):
        return (col(col_name).isNull() | isnan(col(col_name)))
    elif isinstance(dtype, StringType):
        # Optionally include empty string "" as missing:
        return (col(col_name).isNull() | (col(col_name) == ""))
    elif isinstance(dtype, (TimestampType, DateType)):
        return col(col_name).isNull()
    else:
        return col(col_name).isNull()


In [10]:
from pyspark.ml.feature import Imputer
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when, isnan, isnull, count

def analyze_missing_patterns(df):
    """Analyze missing data to determine optimal imputation strategy"""
    missing_stats = {}
    total_rows = df.count()
    
    for col_name in df.columns:
        dtype = dict(df.dtypes)[col_name]  
        missing_count = df.filter(is_missing(col_name,dtype)).count()
        missing_pct = (missing_count / total_rows) * 100
        missing_stats[col_name] = missing_pct
        
    return missing_stats

# Categorize columns by complexity
def categorize_for_imputation(df):
    """Categorize columns for different imputation strategies"""
    missing_stats = analyze_missing_patterns(df)
    
    simple_numeric = []      # <10% missing, numeric
    complex_numeric = []     # >10% missing, numeric  
    categorical_cols = []    # All categorical columns
    time_data_cols = []           # Date/Time columns
    
    for field in df.schema.fields:
        col_name = field.name
        missing_pct = missing_stats.get(col_name, 0)
        
        if isinstance(field.dataType, (DoubleType, IntegerType)):
            if missing_pct < 10:
                simple_numeric.append(col_name)
            else:
                complex_numeric.append(col_name)
        elif isinstance(field.dataType, (TimestampType, DateType)):
            time_data_cols.append(col_name)
        else:
            categorical_cols.append(col_name)
    
    return simple_numeric, complex_numeric, categorical_cols, time_data_cols


In [11]:
def apply_pyspark_imputation(df, simple_numeric_cols):
    """Apply PySpark Imputer for simple numeric imputation"""
    
    pipeline_stages = []
    
    if simple_numeric_cols:
        # Mean imputation for normally distributed data
        mean_imputer = Imputer(
            inputCols=simple_numeric_cols,
            outputCols=[f"{col}_imputed" for col in simple_numeric_cols],
            strategy="mean"
        )
        pipeline_stages.append(mean_imputer)
        
        # Create and execute pipeline
        if pipeline_stages:
            simple_pipeline = Pipeline(stages=pipeline_stages)
            return simple_pipeline.fit(df).transform(df)
    
    return df


In [12]:
def handle_categorical_imputation(df, categorical_cols):
    """Mode imputation for categorical variables"""
    
    df_result = df
    
    for col_name in categorical_cols:
        # Get mode (most frequent value)
        mode_value = (df_result
                     .filter(col(col_name).isNotNull())
                     .groupBy(col_name)
                     .count()
                     .orderBy(col("count").desc())
                     .first())
        
        if mode_value:
            df_result = df_result.fillna({col_name: mode_value[0]})
    
    return df_result


In [13]:
def chunked_mice_imputation(spark_df, complex_numeric_cols, chunk_size=500000):
    """
    Memory-efficient MICE imputation for large datasets
    Processes data in chunks to manage 6GB size
    """
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from sklearn.ensemble import RandomForestRegressor
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, lit
    

    
    window = Window.orderBy(lit(1))
    df_indexed = spark_df.withColumn("chunk_id", 
                                     ((row_number().over(window) - 1) / chunk_size).cast("integer"))
    
    # Process each chunk
    chunk_ids = [row.chunk_id for row in df_indexed.select("chunk_id").distinct().collect()]
    imputed_chunks = []
    
    for chunk_id in chunk_ids:
        print(f"Processing MICE chunk {chunk_id + 1}/{len(chunk_ids)}")
        
        # Extract and process chunk
        chunk_df = df_indexed.filter(col("chunk_id") == chunk_id).drop("chunk_id")
        chunk_pandas = chunk_df.toPandas()
        
        # Apply MICE only to complex numeric columns
        if complex_numeric_cols:
            mice_imputer = IterativeImputer(
                estimator=RandomForestRegressor(n_estimators=10, random_state=42),
                max_iter=2,
                random_state=42
            )
            
            # Impute missing values
            imputed_values = mice_imputer.fit_transform(chunk_pandas[complex_numeric_cols])
            chunk_pandas[complex_numeric_cols] = imputed_values
        
        # Convert back to Spark
        imputed_chunk = SparkSession_init.createDataFrame(chunk_pandas)
        imputed_chunks.append(imputed_chunk)
    
    # Union all chunks
    final_df = imputed_chunks[0]
    for chunk in imputed_chunks[1:]:
        final_df = final_df.union(chunk)
    
    return final_df


In [14]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, isnan, isnull, coalesce, lit, 
    current_date, current_timestamp, to_date, to_timestamp,
    last, first, date_add, date_sub, datediff,
    avg as spark_avg, unix_timestamp, from_unixtime,
    lag, lead, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, TimestampType
from typing import List, Optional, Union
import datetime

def impute_date_timestamp_columns(
    df: DataFrame,
    columns: Optional[Union[List[str], str]] = None,
    strategy: str = 'forward_fill',
    partition_cols: Optional[List[str]] = None,
    order_col: Optional[str] = None,
    default_date: Optional[str] = None,
    max_gap_days: Optional[int] = None
) -> DataFrame:
    """
    Comprehensive function to impute missing values in date and timestamp columns.
    
    Parameters:
    -----------
    df : DataFrame
        Input PySpark DataFrame
    columns : str or List[str], optional
        Specific date/timestamp columns to impute. If None, auto-detects all date/timestamp columns
    strategy : str, default 'forward_fill'
        Imputation strategy. Options:
        - 'forward_fill': Fill with last valid date (LOCF - Last Observation Carried Forward)
        - 'backward_fill': Fill with next valid date (NOCB - Next Observation Carried Backward)  
        - 'mean_date': Fill with mean date of the column
        - 'median_date': Fill with median date of the column
        - 'current_date': Fill with current date/timestamp
        - 'linear_interpolation': Linear interpolation between surrounding dates
        - 'custom_date': Fill with a specified default date
        - 'auto': Combination of forward fill, then backward fill for remaining nulls
    partition_cols : List[str], optional
        Columns to partition by (e.g., user_id, group_id for time series data)
    order_col : str, optional
        Column to order by for time-based operations (required for forward/backward fill)
    default_date : str, optional
        Default date string for 'custom_date' strategy (format: 'YYYY-MM-DD' or 'YYYY-MM-DD HH:MM:SS')
    max_gap_days : int, optional
        Maximum gap in days to fill (prevents filling very large gaps)
    
    Returns:
    --------
    DataFrame: DataFrame with imputed date/timestamp values
    """
    
    # Auto-detect date/timestamp columns if not specified
    if columns is None:
        columns = [field.name for field in df.schema.fields 
                  if isinstance(field.dataType, (DateType, TimestampType))]
    elif isinstance(columns, str):
        columns = [columns]
    
    if not columns:
        print("No date/timestamp columns found or specified.")
        return df
    
    print(f"Imputing columns: {columns} using strategy: {strategy}")
    
    result_df = df
    
    for column in columns:
        # Check if column exists
        if column not in df.columns:
            print(f"Warning: Column '{column}' not found in DataFrame")
            continue
            
        # Determine if it's a date or timestamp column
        col_type = dict(df.dtypes)[column]
        is_timestamp = col_type == 'timestamp'
        
        # Apply the specified strategy
        if strategy == 'forward_fill':
            result_df = _forward_fill_dates(result_df, column, partition_cols, order_col, max_gap_days)
            
        elif strategy == 'backward_fill':
            result_df = _backward_fill_dates(result_df, column, partition_cols, order_col, max_gap_days)
            
        elif strategy == 'mean_date':
            result_df = _mean_date_imputation(result_df, column, is_timestamp)
            
        elif strategy == 'median_date':
            result_df = _median_date_imputation(result_df, column, is_timestamp)
            
        elif strategy == 'current_date':
            result_df = _current_date_imputation(result_df, column, is_timestamp)
            
        elif strategy == 'linear_interpolation':
            result_df = _linear_interpolation_dates(result_df, column, partition_cols, order_col)
            
        elif strategy == 'custom_date':
            result_df = _custom_date_imputation(result_df, column, default_date, is_timestamp)
            
        elif strategy == 'auto':
            # First forward fill, then backward fill for remaining nulls
            result_df = _forward_fill_dates(result_df, column, partition_cols, order_col, max_gap_days)
            result_df = _backward_fill_dates(result_df, column, partition_cols, order_col, max_gap_days)
            
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
    
    return result_df

# Helper functions for different imputation strategies

def _forward_fill_dates(df: DataFrame, column: str, partition_cols: Optional[List[str]], 
                       order_col: Optional[str], max_gap_days: Optional[int]) -> DataFrame:
    """Forward fill (LOCF) implementation for dates"""
    if order_col is None:
        raise ValueError("order_col is required for forward_fill strategy")
    
    # Create window specification
    if partition_cols:
        window_spec = Window.partitionBy(*partition_cols).orderBy(order_col)
    else:
        window_spec = Window.orderBy(order_col)
    
    # Forward fill using last non-null value
    filled_col = last(col(column), ignorenulls=True).over(window_spec)
    
    if max_gap_days:
        # Only fill if gap is within max_gap_days
        last_valid_date = last(col(order_col), ignorenulls=True).over(
            window_spec.rowsBetween(Window.unboundedPreceding, -1)
        )
        gap_condition = datediff(col(order_col), last_valid_date) <= max_gap_days
        filled_col = when(gap_condition, filled_col).otherwise(col(column))
    
    return df.withColumn(column, coalesce(col(column), filled_col))

def _backward_fill_dates(df: DataFrame, column: str, partition_cols: Optional[List[str]], 
                        order_col: Optional[str], max_gap_days: Optional[int]) -> DataFrame:
    """Backward fill (NOCB) implementation for dates"""
    if order_col is None:
        raise ValueError("order_col is required for backward_fill strategy")
    
    # Create window specification for backward looking
    if partition_cols:
        window_spec = Window.partitionBy(*partition_cols).orderBy(col(order_col).desc())
    else:
        window_spec = Window.orderBy(col(order_col).desc())
    
    # Backward fill using last non-null value (in reverse order)
    filled_col = last(col(column), ignorenulls=True).over(window_spec)
    
    if max_gap_days:
        # Only fill if gap is within max_gap_days  
        next_valid_date = last(col(order_col), ignorenulls=True).over(
            window_spec.rowsBetween(Window.unboundedPreceding, -1)
        )
        gap_condition = datediff(next_valid_date, col(order_col)) <= max_gap_days
        filled_col = when(gap_condition, filled_col).otherwise(col(column))
    
    return df.withColumn(column, coalesce(col(column), filled_col))

def _mean_date_imputation(df: DataFrame, column: str, is_timestamp: bool) -> DataFrame:
    """Fill with mean date/timestamp"""
    # Convert to unix timestamp, calculate mean, convert back
    mean_unix = df.select(spark_avg(unix_timestamp(col(column)))).collect()[0][0]
    
    if mean_unix is not None:
        if is_timestamp:
            mean_value = from_unixtime(lit(mean_unix))
        else:
            mean_value = to_date(from_unixtime(lit(mean_unix)))
        
        return df.withColumn(column, coalesce(col(column), mean_value))
    
    return df

def _median_date_imputation(df: DataFrame, column: str, is_timestamp: bool) -> DataFrame:
    """Fill with median date/timestamp (approximate using percentile)"""
    # Get median using approximate percentile
    median_unix = df.select(col(column)).na.drop().select(
        unix_timestamp(col(column)).alias("ts")
    ).approxQuantile("ts", [0.5], 0.01)[0]
    
    if median_unix is not None:
        if is_timestamp:
            median_value = from_unixtime(lit(median_unix))
        else:
            median_value = to_date(from_unixtime(lit(median_unix)))
        
        return df.withColumn(column, coalesce(col(column), median_value))
    
    return df

def _current_date_imputation(df: DataFrame, column: str, is_timestamp: bool) -> DataFrame:
    """Fill with current date/timestamp"""
    if is_timestamp:
        current_value = current_timestamp()
    else:
        current_value = current_date()
    
    return df.withColumn(column, coalesce(col(column), current_value))

def _custom_date_imputation(df: DataFrame, column: str, default_date: str, is_timestamp: bool) -> DataFrame:
    """Fill with custom specified date"""
    if default_date is None:
        raise ValueError("default_date must be specified for custom_date strategy")
    
    if is_timestamp:
        if len(default_date.split()) == 1:  # Only date provided
            default_date += " 00:00:00"
        default_value = lit(default_date).cast(TimestampType())
    else:
        default_value = lit(default_date.split()[0]).cast(DateType())  # Take only date part
    
    return df.withColumn(column, coalesce(col(column), default_value))

def _linear_interpolation_dates(df: DataFrame, column: str, partition_cols: Optional[List[str]], 
                               order_col: Optional[str]) -> DataFrame:
    """Linear interpolation for dates"""
    if order_col is None:
        raise ValueError("order_col is required for linear_interpolation strategy")
    
    # Create window specification
    if partition_cols:
        window_spec = Window.partitionBy(*partition_cols).orderBy(order_col)
    else:
        window_spec = Window.orderBy(order_col)
    
    # Add row numbers for interpolation calculation
    df_with_rn = df.withColumn("rn", row_number().over(window_spec))
    df_with_rn = df_with_rn.withColumn("rn_not_null", 
                                       when(col(column).isNotNull(), col("rn")))
    
    # Get last valid value and row number before current row
    window_start = window_spec.rowsBetween(Window.unboundedPreceding, -1)
    df_with_rn = df_with_rn.withColumn("start_val", last(col(column), True).over(window_start))
    df_with_rn = df_with_rn.withColumn("start_rn", last("rn_not_null", True).over(window_start))
    
    # Get next valid value and row number after current row  
    window_end = window_spec.rowsBetween(0, Window.unboundedFollowing)
    df_with_rn = df_with_rn.withColumn("end_val", first(col(column), True).over(window_end))
    df_with_rn = df_with_rn.withColumn("end_rn", first("rn_not_null", True).over(window_end))
    
    # Calculate interpolated value
    df_with_rn = df_with_rn.withColumn("diff_rn", col("end_rn") - col("start_rn"))
    df_with_rn = df_with_rn.withColumn("curr_rn", col("diff_rn") - (col("end_rn") - col("rn")))
    
    # Linear interpolation formula for unix timestamps
    start_unix = unix_timestamp("start_val")
    end_unix = unix_timestamp("end_val")
    interpolated_unix = start_unix + (end_unix - start_unix) / col("diff_rn") * col("curr_rn")
    
    # Convert back to original date/timestamp type
    col_type = dict(df.dtypes)[column]
    if col_type == 'timestamp':
        interpolated_value = from_unixtime(interpolated_unix)
    else:
        interpolated_value = to_date(from_unixtime(interpolated_unix))
    
    # Apply interpolation only where values are null
    result_df = df_with_rn.withColumn(
        column, 
        when(col(column).isNull(), interpolated_value).otherwise(col(column))
    )
    
    # Clean up helper columns
    return result_df.drop("rn", "rn_not_null", "start_val", "end_val", 
                         "start_rn", "end_rn", "diff_rn", "curr_rn")

# Additional utility function to check missing values in date columns
def analyze_date_missing_values(df: DataFrame) -> DataFrame:
    """Analyze missing values in date/timestamp columns"""
    date_cols = [field.name for field in df.schema.fields 
                if isinstance(field.dataType, (DateType, TimestampType))]
    
    if not date_cols:
        print("No date/timestamp columns found in DataFrame")
        return None
    
    from pyspark.sql.functions import sum as spark_sum, count, round as spark_round
    
    # Calculate missing statistics for each date column
    total_count = df.count()
    
    missing_stats = []
    for col_name in date_cols:
        null_count = df.filter(col(col_name).isNull()).count()
        missing_percentage = (null_count / total_count) * 100
        missing_stats.append((col_name, null_count, missing_percentage))
    
    # Create summary DataFrame
    summary_df = SparkSession_init.createDataFrame(
        missing_stats, 
        ["column_name", "null_count", "missing_percentage"]
    )
    
    return summary_df

In [ ]:
def hybrid_imputation_pipeline(spark_df):
    """
    Complete hybrid imputation strategy for 6GB mixed-datatype dataset
    """
    
    print("Starting Hybrid Imputation Pipeline...")
    
    # Step 1: Analyze and categorize
    simple_numeric, complex_numeric, categorical,time_data = categorize_for_imputation(spark_df)
    
    print(f"Simple numeric columns: {len(simple_numeric)}")
    print(f"Complex numeric columns: {len(complex_numeric)}")
    print(f"Categorical columns: {len(categorical)}")
    print(f"Time Data  columns: {len(time_data)}")
    
    # Step 2: PySpark Imputer for simple cases
    df_stage1 = apply_pyspark_imputation(spark_df, simple_numeric)
    print("✓ PySpark Imputer completed")
    
    # Step 3: Categorical imputation
    df_stage2 = handle_categorical_imputation(df_stage1, categorical)
    print("✓ Categorical imputation completed")
    
    # Step 4: MICE for complex cases
    if complex_numeric:
        df_final = chunked_mice_imputation(df_stage2, complex_numeric, chunk_size=50000)
        print("✓ MICE imputation completed")
    else:
        df_final = df_stage2
    
    return df_final

# Usage with your dataset
imputed_ML_data = hybrid_imputation_pipeline(ML_data_parquet)


Starting Hybrid Imputation Pipeline...


Simple numeric columns: 29
Complex numeric columns: 1
Categorical columns: 14
Time Data  columns: 1


✓ PySpark Imputer completed


✓ Categorical imputation completed


25/07/27 06:24:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/27 06:24:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/27 06:24:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/27 06:24:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/27 06:24:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/27 06:24:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/27 0

...

25/07/27 06:58:23 WARN TaskMemoryManager: Failed to allocate a page (8388592 bytes), try again.


[2340.778s][warning][gc,alloc] Executor task launch worker for task 0.0 in stage 200.0 (TID 1417): Retried waiting for GCLocker too often allocating 131074 words


25/07/27 06:59:17 ERROR Executor: Exception in task 0.0 in stage 200.0 (TID 1417)
java.lang.OutOfMemoryError: Java heap space
25/07/27 06:59:17 ERROR SparkUncaughtExceptionHandler: Uncaught exception in thread Thread[Executor task launch worker for task 0.0 in stage 200.0 (TID 1417),5,main]
java.lang.OutOfMemoryError: Java heap space
25/07/27 06:59:17 WARN TaskSetManager: Lost task 0.0 in stage 200.0 (TID 1417) (25bb953e3acb executor driver): java.lang.OutOfMemoryError: Java heap space

25/07/27 06:59:17 ERROR TaskSetManager: Task 0 in stage 200.0 failed 1 times; aborting job


ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
output_path = "/home/john/volume_folder/imputed_ml_data_test.parquet"
imputed_ML_data.write.parquet(output_path)

In [39]:
# SparkSession_init.stop()
SparkSession_init.stop()